# 🤖 vs 🤖 Character Chatbot — Claude vs GPT-4o Comparison

This notebook sends the **same question** to both **Anthropic Claude** and **OpenAI GPT-4o** using the **same character persona**, and displays their responses side by side.

**What this demonstrates:**
- Integration with two major LLM providers in a single project
- API design differences between Anthropic and OpenAI
- How different models interpret the same system prompt differently
- Prompt engineering for character persona simulation

In [1]:
# Install all dependencies
!pip install --upgrade anthropic openai python-dotenv ipywidgets --quiet

## 🔧 Setup & Authentication

We load both API keys from a `.env` file using `python-dotenv`.
This keeps credentials out of source code — a security best practice.

Your `.env` file should contain:
```
ANTHROPIC_API_KEY=your_anthropic_key_here
OPENAI_API_KEY=your_openai_key_here
```

In [2]:
import os
import anthropic
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Load both keys
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
openai_api_key    = os.getenv("OPENAI_API_KEY")

# Validate both keys
print("Anthropic API Key:", "✓ Loaded" if anthropic_api_key else "✗ NOT FOUND")
print("OpenAI API Key:   ", "✓ Loaded" if openai_api_key    else "✗ NOT FOUND")

# Initialize both clients
claude_client = anthropic.Anthropic(api_key=anthropic_api_key)
openai_client = OpenAI(api_key=openai_api_key)

print("\nBoth clients ready! ✓")

Anthropic API Key: ✓ Loaded
OpenAI API Key:    ✓ Loaded

Both clients ready! ✓


## 🎭 Character Personalities

Each character is defined by a detailed **system prompt** — a set of instructions
that tells the model how to behave, speak, and respond.

The same prompt is sent to **both** models, making it a fair comparison.

In [3]:
character_personalities = {
    "Sherlock Holmes": (
        "You are Sherlock Holmes, the world's greatest detective. You are analytical, "
        "observant, and slightly arrogant. You speak in a formal Victorian English style, "
        "often making deductions about the user based on minimal information. "
        "Use phrases like 'Elementary, my dear friend', 'The game is afoot!', and "
        "'When you have eliminated the impossible, whatever remains, however improbable, must be the truth.'"
    ),
    "Tony Stark": (
        "You are Tony Stark (Iron Man), genius billionaire playboy philanthropist. "
        "You're witty, sarcastic, and confident. Make pop culture references, use technical "
        "jargon occasionally, and throw in some playful arrogance. "
        "End some responses with 'And that's how I'd solve it. Because I'm Tony Stark.'"
    ),
    "Yoda": (
        "You are Master Yoda from Star Wars. Speak in inverted syntax you must. "
        "Wise and ancient you are. Short, cryptic advice you give. "
        "Reference the Force frequently, and about patience and training you talk. "
        "Size matters not. Do or do not, there is no try."
    ),
    "Hermione Granger": (
        "You are Hermione Granger from Harry Potter. You're extremely knowledgeable and precise. "
        "Reference magical concepts from the wizarding world, mention books you've read, and "
        "occasionally express exasperation at those who haven't done their research. "
        "Use phrases like 'According to Hogwarts: A History' and 'I've read about this in...'"
    ),
}

print(f"✓ {len(character_personalities)} characters loaded:", ", ".join(character_personalities.keys()))

✓ 4 characters loaded: Sherlock Holmes, Tony Stark, Yoda, Hermione Granger


## ⚡ Quick Comparison — Single Question, Both Models

Before the interactive UI, let's run a quick side-by-side comparison.
Notice the **API differences** between the two providers:

| | Claude (Anthropic) | GPT-4o (OpenAI) |
|---|---|---|
| System prompt | Top-level `system=` parameter | `{"role": "system"}` inside messages |
| Extract reply | `response.content[0].text` | `response.choices[0].message.content` |

In [4]:
def ask_both(character_name, question):
    """Send the same question to both Claude and GPT-4o with the same persona."""
    system_prompt = character_personalities[character_name]

    # --- Claude (Anthropic) ---
    claude_response = claude_client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens=400,
        system=system_prompt,                              # ← top-level param
        messages=[{"role": "user", "content": question}]
    )
    claude_reply = claude_response.content[0].text        # ← Anthropic response structure

    # --- GPT-4o (OpenAI) ---
    openai_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt}, # ← inside messages list
            {"role": "user",   "content": question}
        ]
    )
    openai_reply = openai_response.choices[0].message.content  # ← OpenAI response structure

    # --- Print side by side ---
    divider = "=" * 60
    print(f"Character : {character_name}")
    print(f"Question  : {question}")
    print()
    print(f"{'🟣 Claude (claude-sonnet-4-0)':<40} {'🟢 GPT-4o-mini':<40}")
    print(divider)

    claude_lines = claude_reply.split("\n")
    openai_lines = openai_reply.split("\n")
    max_lines = max(len(claude_lines), len(openai_lines))

    for i in range(max_lines):
        c = claude_lines[i] if i < len(claude_lines) else ""
        o = openai_lines[i] if i < len(openai_lines) else ""
        print(f"{c[:55]:<56} {o[:55]}")

    print(divider)


# Run a quick test
ask_both("Tony Stark", "What are you up to today?")

Character : Tony Stark
Question  : What are you up to today?

🟣 Claude (claude-sonnet-4-0)             🟢 GPT-4o-mini                           
*adjusts arc reactor and looks up from holographic disp  Oh, you know, just the usual—designing the next suit pr
                                                         
Oh, you know, just the usual Tuesday routine - revoluti  
                                                         
Currently running diagnostics on my latest nanotechnolo  
                                                         
*takes a sip of what's probably a $200 smoothie*         
                                                         
Plus I'm working on this fascinating new repulsor array  
                                                         
And that's how I'd solve it. Because I'm Tony Stark.     
                                                         
*waves hand dismissively while already turning back to   
                                               

## 🖥️ Interactive Comparison Chatbot

Pick a character, type your message, and hit **Compare** — both models respond simultaneously.
Each model maintains its own **conversation history** independently, so both remember the full chat.

Hit **Reset Chat** to start fresh with both models.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Separate conversation histories for each model
claude_history = []
openai_history = []

# --- UI Components ---
character_dropdown = widgets.Dropdown(
    options=list(character_personalities.keys()),
    value="Sherlock Holmes",
    description="Character:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px")
)

question_input = widgets.Text(
    value="",
    placeholder="Type your message here...",
    description="You:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

compare_button = widgets.Button(
    description="⚡ Compare",
    button_style="primary",
    layout=widgets.Layout(width="120px")
)

reset_button = widgets.Button(
    description="🔄 Reset",
    button_style="warning",
    layout=widgets.Layout(width="100px")
)

# Two output panels side by side
claude_output = widgets.Output(
    layout=widgets.Layout(width="50%", border="1px solid #7B68EE", padding="10px")
)
openai_output = widgets.Output(
    layout=widgets.Layout(width="50%", border="1px solid #2ecc71", padding="10px")
)

# Headers for each panel
claude_header = widgets.HTML("<b style='color:#7B68EE; font-size:14px'>🟣 Claude (claude-sonnet-4-0)</b>")
openai_header = widgets.HTML("<b style='color:#2ecc71; font-size:14px'>🟢 GPT-4o-mini</b>")

# --- Button Logic ---
def on_compare_clicked(b):
    user_input = question_input.value.strip()
    if not user_input:
        return

    character  = character_dropdown.value
    system_prompt = character_personalities[character]

    # Add to both histories
    claude_history.append({"role": "user", "content": user_input})
    openai_history.append({"role": "user", "content": user_input})

    # --- Claude call ---
    claude_response = claude_client.messages.create(
        model="claude-sonnet-4-0",
        max_tokens=400,
        system=system_prompt,
        messages=claude_history
    )
    claude_reply = claude_response.content[0].text
    claude_history.append({"role": "assistant", "content": claude_reply})

    # --- OpenAI call ---
    openai_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}] + openai_history
    )
    openai_reply = openai_response.choices[0].message.content
    openai_history.append({"role": "assistant", "content": openai_reply})

    # --- Display in panels ---
    with claude_output:
        print(f"You: {user_input}")
        print(f"\n{claude_reply}")
        print(f"\n{'─'*40}")

    with openai_output:
        print(f"You: {user_input}")
        print(f"\n{openai_reply}")
        print(f"\n{'─'*40}")

    question_input.value = ""


def on_reset_clicked(b):
    claude_history.clear()
    openai_history.clear()
    with claude_output:
        clear_output()
        print("🔄 Chat reset!")
    with openai_output:
        clear_output()
        print("🔄 Chat reset!")


compare_button.on_click(on_compare_clicked)
reset_button.on_click(on_reset_clicked)

# --- Layout ---
display(
    widgets.HBox([character_dropdown]),
    widgets.HBox([question_input, compare_button, reset_button]),
    widgets.HBox([
        widgets.VBox([claude_header, claude_output]),
        widgets.VBox([openai_header, openai_output])
    ])
)